# Male Fly Connectome — multiscale LOD pipeline

Colab workflow for the v0.4 research milestone. It downloads **authoritative MaleCNS v1.0** SWC data, builds topology-preserving LOD bundles, and validates hashes/metrics. Generative AI is not used as a source of biological data.


In [ ]:
!git clone https://github.com/arigony/MaleFlyConnectome.git
%cd MaleFlyConnectome
!pip -q install -r requirements.txt pyarrow


## Choose the build scale
`smoke` is fast. `types` downloads one deterministic body ID per annotated cell type. A full all-neuron build is intentionally not launched by default because it can involve very large storage/network use.


In [ ]:
MODE = 'smoke'  # 'smoke' or 'types'
LIMIT = 200

if MODE == 'smoke':
    cmd = f'python scripts/fetch_malecns_skeletons.py --one-per-type --limit {LIMIT} --output data/skeletons-global'
elif MODE == 'types':
    cmd = 'python scripts/fetch_malecns_skeletons.py --one-per-type --output data/skeletons-global'
else:
    raise ValueError('Use smoke or types in this notebook.')
print(cmd)
!{cmd}


In [ ]:
!python scripts/build_lod.py --input data/skeletons-global --output data/lod-global
!python scripts/validate_lod.py --dir data/lod-global


In [ ]:
import json, pandas as pd
m = json.load(open('data/lod-global/metrics.json'))
rows = []
for n in m['neurons']:
    for level, x in n['levels'].items():
        rows.append({'bodyId': n['bodyId'], 'level': level, **x})
df = pd.DataFrame(rows)
display(df.groupby('level')[['rendered_segments','segment_reduction_pct']].agg(['sum','mean']))


The generated `metrics.json` and `manifest.json` are the basis for publication-grade fidelity and performance tables. Do not interpret an RDP tolerance as microscopy resolution.
